# Banking Analytics — Exploratory Data Analysis

**Project:** Predictive Analytics and Recommendation Systems in Banking

This notebook covers EDA for loan, transaction, and interaction datasets.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

RAW = ROOT / 'data' / 'raw'
sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Generate data if not present
if not (RAW / 'loans.csv').exists():
    from src.generate_data import run
    run()

loans = pd.read_csv(RAW / 'loans.csv')
transactions = pd.read_csv(RAW / 'transactions.csv')
interactions = pd.read_csv(RAW / 'interactions.csv')
products = pd.read_csv(RAW / 'products.csv')

print(loans.shape, transactions.shape, interactions.shape)

## 1. Loan Default Dataset

In [ ]:
loans.head()

In [ ]:
print('Default rate:', loans['repayment_status'].mean())
loans.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(loans['credit_score'], kde=True, ax=axes[0])
axes[0].set_title('Credit Score Distribution')
sns.boxplot(data=loans, x='credit_risk_band', y='debt_to_income_ratio', hue='repayment_status', ax=axes[1])
axes[1].set_title('Debt-to-Income by Risk Band & Default')
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['income', 'credit_score', 'loan_amount', 'debt_to_income_ratio', 'payment_to_income_ratio', 'repayment_status']
sns.heatmap(loans[corr_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Loan Feature Correlation')
plt.show()

## 2. Transaction Data

In [ ]:
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
transactions.groupby('transaction_type')['transaction_amount'].agg(['count', 'mean', 'sum']).round(2)

In [ ]:
monthly = transactions.groupby(transactions['transaction_date'].dt.to_period('M'))['transaction_amount'].sum()
monthly.plot(figsize=(10, 4), title='Monthly Transaction Volume')
plt.ylabel('Total Amount')
plt.show()

## 3. Product Interactions

In [ ]:
interactions['interaction_type'].value_counts().plot(kind='bar', title='Interaction Types', figsize=(8, 4))
plt.ylabel('Count')
plt.show()

In [ ]:
top_products = interactions.merge(products, on='product_id').groupby('product_name').size().sort_values(ascending=False).head(10)
top_products.plot(kind='barh', figsize=(8, 5), title='Top 10 Products by Interactions')
plt.xlabel('Interactions')
plt.show()

## 4. Key Insights

- Higher **debt-to-income** and lower **credit scores** correlate with loan defaults.
- Transaction behavior varies by type (deposits vs withdrawals) — useful for segmentation.
- Interaction data is sparse; hybrid recommendation handles cold-start better than pure CF.